# Compare instances vs simplified instances

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from shapely.plotting import plot_polygon
from tspn_bnb2 import AnnotatedInstance
from tspn_bnb2.core._tspn_bindings import Instance, parse_site_wkt
from tspn_bnb2.misc.paper_style import init_params

init_params()

In [ ]:
from shapely.ops import unary_union
from shapely.validation import make_valid


def extract_polygons(geom):
    if geom.is_empty:
        return []

    if geom.geom_type == "Polygon":
        return [geom]

    if geom.geom_type == "MultiPolygon":
        return list(geom.geoms)

    if geom.geom_type == "GeometryCollection":
        return [g for g in geom.geoms if g.geom_type == "Polygon"]

    return []

## Load both instance sets

In [ ]:
def load_instances(zip_path: str) -> dict[str, AnnotatedInstance]:
    instances = {}
    with ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            if name.endswith(".json"):
                stem = Path(name).stem
                instances[stem] = AnnotatedInstance.model_validate_json(zf.open(name).read())
    return instances


instances = load_instances("../../instances/instances_socg.zip")
simplified_instances = load_instances("../../instances/instances_socg_simplified.zip")

print(f"instances: {len(instances)}")
print(f"simplified_instances: {len(simplified_instances)}")

## Compare instances by unary union symmetric difference

In [ ]:
def count_convex_pieces(ann_inst: AnnotatedInstance) -> int:
    sites = [parse_site_wkt(p.wkt) for p in ann_inst.polygons]
    inst = Instance(sites, False)
    total = 0
    for i in range(len(inst)):
        decomp = inst[i].decomposition()
        num_pieces = len(decomp) if decomp else 1
        total += num_pieces
    return total


common_keys = sorted(set(instances) & set(simplified_instances))
only_instances = sorted(set(instances) - set(simplified_instances))
only_simplified = sorted(set(simplified_instances) - set(instances))

print(
    f"Common: {len(common_keys)}, "
    f"only in instances: {len(only_instances)}, "
    f"only in simplified: {len(only_simplified)}"
)

rows = []
for key in common_keys:
    input_geom = make_valid(unary_union(instances[key].polygons)).buffer(0)
    output_geom = make_valid(unary_union(simplified_instances[key].polygons)).buffer(0)

    removed = extract_polygons(input_geom.difference(output_geom))
    added = extract_polygons(output_geom.difference(input_geom))

    is_removed = len(removed) > 0
    is_added = len(added) > 0
    is_same = not is_removed and not is_added

    total_area = input_geom.area
    removed_area = sum(p.area for p in removed)
    added_area = sum(p.area for p in added)

    rows.append(
        {
            "name": key,
            "n": len(instances[key].polygons),
            "polygons_removed": is_removed,
            "polygons_added": is_added,
            "is_same": is_same,
            "total_area": total_area,
            "removed_area": removed_area,
            "added_area": added_area,
            "removed_pct": removed_area / total_area * 100 if total_area > 0 else 0,
            "added_pct": added_area / total_area * 100 if total_area > 0 else 0,
            "convex_pieces_original": count_convex_pieces(instances[key]),
            "convex_pieces_simplified": count_convex_pieces(simplified_instances[key]),
        }
    )

df = pd.DataFrame(rows)
print(f"\nTotal common instances: {len(df)}")
print(f"Same: {df['is_same'].sum()}, Different: {(~df['is_same']).sum()}")

In [ ]:
round(df["convex_pieces_original"] - df["convex_pieces_simplified"]).mean()

## Summary by polygon count

In [ ]:
summary = (
    df.groupby("n")
    .agg(
        total=("is_same", "count"),
        same=("is_same", "sum"),
        has_added_polygons=("polygons_added", "sum"),
        has_removed_polygons=("polygons_removed", "sum"),
        different=("is_same", lambda x: (~x).sum()),
    )
    .reset_index()
)
summary["same"] = summary["same"].astype(int)
summary["different"] = summary["different"].astype(int)
summary

## Visualize differing instances

In [ ]:
differing = df[~df["is_same"]].sample(n=min(6, (~df["is_same"]).sum()), random_state=42)

for _, row in differing.iterrows():
    name = row["name"]
    a_polys = instances[name].polygons
    b_polys = simplified_instances[name].polygons

    fig, axs = plt.subplots(ncols=3, figsize=(6, 3))
    for ax in axs.flat:
        ax.set_axis_off()
        ax.set_aspect("equal")

    # Left: original version
    for polygon in a_polys:
        plot_polygon(polygon=polygon, ax=axs[0], add_points=False, color="grey")

    # Right: simplified version
    for polygon in b_polys:
        plot_polygon(polygon=polygon, ax=axs[2], add_points=False, color="grey")

    # Middle: overlay with differences
    for polygon in a_polys:
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="grey")

    input_geom = make_valid(unary_union(a_polys)).buffer(0)
    output_geom = make_valid(unary_union(b_polys)).buffer(0)

    removed = extract_polygons(input_geom.difference(output_geom))
    added = extract_polygons(output_geom.difference(input_geom))

    for polygon in removed:
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="red", alpha=0.3)

    for polygon in added:
        plot_polygon(polygon=polygon, ax=axs[1], add_points=False, color="green", alpha=0.3)

    axs[0].set_title("Original")
    axs[1].set_title("Differences")
    axs[2].set_title("Simplified")
    fig.suptitle(name)
    fig.tight_layout()

## Area and convex piece analysis

In [ ]:
# Melt removed/added percentages into long form for seaborn
area_df = df[["name", "n", "removed_pct", "added_pct"]].melt(
    id_vars=["name", "n"],
    value_vars=["removed_pct", "added_pct"],
    var_name="type",
    value_name="area_pct",
)
area_df["type"] = area_df["type"].map({"removed_pct": "Removed", "added_pct": "Added"})
area_df = (
    area_df[area_df["area_pct"] > 0].sort_values("area_pct", ascending=False).reset_index(drop=True)
)

fig, axs = plt.subplots(ncols=2, figsize=(5.788, 2.5))

# Left: removed and added area percentages
sns.stripplot(
    data=area_df,
    x="n",
    y="area_pct",
    hue="type",
    dodge=True,
    size=2.5,
    alpha=0.7,
    ax=axs[0],
)
axs[0].set_xlabel("$n$")
axs[0].set_ylabel("Area change (\\%)")
axs[0].set_title("Area changed by simplification")
axs[0].legend(title="Type")

# Right: convex pieces comparison
sns.scatterplot(
    data=df[~df["is_same"]],
    x="convex_pieces_original",
    y="convex_pieces_simplified",
    s=10,
    alpha=0.5,
    ax=axs[1],
)
max_val = max(df["convex_pieces_original"].max(), df["convex_pieces_simplified"].max())
axs[1].plot([0, max_val], [0, max_val], "k--", lw=0.5, label="$y = x$")
axs[1].set_xlabel("Convex pieces (original)")
axs[1].set_ylabel("Convex pieces (simplified)")
axs[1].set_title("Convex decomposition complexity")
axs[1].legend()

fig.tight_layout()
fig.savefig("simplification.pdf", facecolor="white")